# K-Means Robustness Decision

This notebook checks whether K-Means with `k=5` is a reasonable final cluster count for the compact RobustScaler modelling input. It does not change preprocessing or use basket data.


## Decision Logic

1. Compare K-Means results for several values of `k`.
2. Check whether the selected `k=5` result is stable across random seeds.
3. Save only the compact evidence tables needed for the project state and final report.


## Imports


In [ ]:
import sys

import matplotlib.pyplot as plt
import pandas as pd

sys.path.append("../src")

from clustering import (
    calculate_seed_stability,
    compare_kmeans_k_values,
    split_customer_features,
    validate_clustering_input,
)


## Load Final Modelling Input

The input is the final compact business feature set scaled with `RobustScaler`.


In [ ]:
selected_model_features = pd.read_csv("../data/processed/selected_model_features.csv")
validate_clustering_input(selected_model_features)

customer_ids, X = split_customer_features(selected_model_features)

input_check = pd.DataFrame({
    "check": [
        "rows",
        "unique customer_id",
        "model features",
        "missing values",
        "duplicated customer_id",
    ],
    "value": [
        len(selected_model_features),
        selected_model_features["customer_id"].nunique(),
        X.shape[1],
        selected_model_features.isna().sum().sum(),
        selected_model_features["customer_id"].duplicated().sum(),
    ],
})
input_check


## Compare Cluster Counts

`k=2` and `k=3` have stronger separation metrics, but they create very broad segments. `k=5` is checked as a more useful segmentation level because it gives more actionable groups while keeping acceptable balance.


In [ ]:
k_metrics = compare_kmeans_k_values(
    X,
    k_values=range(2, 11),
    random_state=42,
    n_init=50,
    sample_size=10000,
)

k_metrics = k_metrics[[
    "k",
    "random_state",
    "n_features",
    "inertia",
    "silhouette_score",
    "calinski_harabasz_score",
    "davies_bouldin_score",
    "min_cluster_size",
    "max_cluster_size",
    "min_cluster_percentage",
    "max_cluster_percentage",
]]

k_metrics.to_csv("../outputs/segmentation_robustness_metrics.csv", index=False)
k_metrics


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].plot(k_metrics["k"], k_metrics["silhouette_score"], marker="o")
axes[0].axvline(5, color="black", linestyle="--", linewidth=1)
axes[0].set_title("Silhouette")
axes[0].set_xlabel("k")

axes[1].plot(k_metrics["k"], k_metrics["davies_bouldin_score"], marker="o")
axes[1].axvline(5, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Davies-Bouldin")
axes[1].set_xlabel("k")

axes[2].plot(k_metrics["k"], k_metrics["max_cluster_percentage"], marker="o", label="Largest cluster")
axes[2].plot(k_metrics["k"], k_metrics["min_cluster_percentage"], marker="o", label="Smallest cluster")
axes[2].axvline(5, color="black", linestyle="--", linewidth=1)
axes[2].set_title("Cluster Balance")
axes[2].set_xlabel("k")
axes[2].legend()

plt.tight_layout()
plt.show()


## Seed Stability For k=5

The same final feature input is refit with several random seeds. High adjusted Rand index values mean the customer assignments are not highly dependent on one random initialization.


In [ ]:
seed_stability = calculate_seed_stability(
    X,
    k=5,
    seeds=[0, 21, 42, 99, 123],
    n_init=50,
)
seed_stability.to_csv("../outputs/segmentation_seed_stability.csv", index=False)
seed_stability


## Recommendation


In [ ]:
k5 = k_metrics.loc[k_metrics["k"] == 5].iloc[0]
seed_summary = seed_stability["adjusted_rand_index"].agg(["min", "mean", "max"])

recommendation = pd.DataFrame([
    {
        "candidate": "KMeans_k5_compact_robust",
        "evidence_for": (
            f"k=5 gives five usable segments with silhouette {k5['silhouette_score']:.3f}, "
            f"Davies-Bouldin {k5['davies_bouldin_score']:.3f}, and minimum cluster share "
            f"{k5['min_cluster_percentage']:.2f}%."
        ),
        "evidence_against": (
            f"The largest cluster is broad at {k5['max_cluster_percentage']:.2f}% of customers; "
            "k=2 and k=3 have stronger separation metrics but are less useful for action."
        ),
        "seed_stability": (
            f"ARI range {seed_summary['min']:.3f} to {seed_summary['max']:.3f}, "
            f"mean {seed_summary['mean']:.3f}."
        ),
        "recommendation": "Keep K-Means k=5 as the final segmentation level.",
    }
])

recommendation.to_csv("../outputs/segmentation_robustness_recommendation.csv", index=False)
recommendation


## Conclusion

K-Means `k=5` is not chosen because it maximizes every internal metric. It is chosen because it balances separation, interpretability, and actionability better than the very broad lower-`k` alternatives. The largest cluster should be discussed as a broad segment in the final report.
